# Phase 1 — Fare and weather modeling

Run the cells in order. The [English report](../reports/en/phase-1-fare-and-weather-modeling.md)
describes the methods and the original saved results. Obtain the input CSV using the
[data instructions](../data/README.md).


In [ ]:
import pandas as pd

from pathlib import Path

# Jupyter may start the kernel in notebooks/ or at the repository root.
data_dir = Path("data") if Path("data").is_dir() else Path("../data")
data_file = data_dir / "rideshare_kaggle.csv"
if not data_file.is_file():
    raise FileNotFoundError(f"Place the Kaggle CSV at {data_file}. See data/README.md.")

# Use the 39 columns documented in the original course notebook.
course_columns = [
    'id', 'hour', 'day', 'month', 'source',
    'destination', 'cab_type', 'product_id', 'name', 'price',
    'distance', 'surge_multiplier', 'latitude', 'longitude', 'temperature',
    'apparentTemperature', 'short_summary', 'long_summary', 'precipIntensity', 'precipProbability',
    'humidity', 'windSpeed', 'windGust', 'visibility', 'temperatureHigh',
    'temperatureLow', 'apparentTemperatureHigh', 'apparentTemperatureLow', 'dewPoint', 'pressure',
    'windBearing', 'cloudCover', 'uvIndex', 'ozone', 'moonPhase',
    'temperatureMin', 'temperatureMax', 'apparentTemperatureMin', 'apparentTemperatureMax',
]


# Q1

### Q1_1

In [ ]:
df = pd.read_csv(data_file, usecols=course_columns)
df.info()
df.isnull().sum()
df.describe().T

In [ ]:
df_clean= df.dropna(subset=['price']).copy()

### Q1_2

In [ ]:
# 1) What origins and destinations are recorded?
origins = df_clean['source'].unique()
destinations = df_clean['destination'].unique()

print("Origins (sources):")
print(origins)
print("\nDestinations:")
print(destinations)

# 2) Which routes have the highest number of trips?
route_counts = (
    df.groupby(['source', 'destination'])
      .size()
      .reset_index(name='num_trips')
      .sort_values('num_trips', ascending=False)
)

print("\nTop routes by number of trips:")
print(route_counts.head(10))   # top 10 routes

# 3) Is there any location that has only outgoing trips and no incoming trips?
origins_set = set(df['source'])
destinations_set = set(df['destination'])

only_outgoing = origins_set - destinations_set

if only_outgoing:
    print("\nLocations that have trips starting from them but no trips going to them:")
    print(only_outgoing)
else:
    print("\nNo location has only outgoing trips; every origin also appears at least once as a destination.")

### Q1_3

In [ ]:
# Check how many unique product_ids exist
unique_ids = df_clean['product_id'].nunique()

# Check how many unique product names exist
unique_names = df_clean['name'].nunique()

print("Unique product_id:", unique_ids)
print("Unique name:", unique_names)

# Check if each product_id always maps to exactly one name
mapping_check = df_clean.groupby('product_id')['name'].nunique()

print("\nNumber of names per product_id:")
print(mapping_check.value_counts())   # how many IDs have 1 name, 2 names, etc.

# Check if the mapping is one-to-one (bijective)
if (mapping_check == 1).all():
    print("\nResult: Each product_id corresponds to exactly ONE name.")
    print("→ So we can safely use only one of the columns.")
else:
    print("\nResult: Some product_ids map to multiple names.")
    print("→ We must keep both product_id and name.")


In [ ]:
df_clean = df_clean.drop(columns=['product_id'])
df_clean.info()

### Q1_4

In [ ]:
# Unique trip names for Uber
uber_names = df_clean[df_clean['cab_type'] == 'Uber']['name'].unique()

# Unique trip names for Lyft
lyft_names = df_clean[df_clean['cab_type'] == 'Lyft']['name'].unique()

print("Uber trip names:")
print(uber_names)

print("\nLyft trip names:")
print(lyft_names)


### Q1_5

In [ ]:
import pandas as pd

df_encoded = pd.get_dummies(df_clean, columns=['name'], prefix='name', drop_first=True)
df_final = df_encoded.copy()
boolean_cols = [col for col in df_final.columns if col.startswith('name_')]
df_final[boolean_cols] = df_final[boolean_cols].astype(int)

# Q2

### Q2_1

In [ ]:
from scipy.stats import f_oneway

# Group the price values according to each 'name' category
# This creates a list: [prices_for_name1, prices_for_name2, ...]
groups = [group['price'].values for name, group in df_clean.groupby('name')]

# Perform one-way ANOVA to check if mean price differs between names
stat, p_value = f_oneway(*groups)

print("F-statistic:", stat)
print("p-value:", p_value)

# Interpretation based on the p-value
if p_value < 0.05:
    print("\nResult: There is a statistically significant difference in price among different 'name' categories.")
else:
    print("\nResult: There is NO statistically significant difference in price among the 'name' categories.")

### Q2_2 & Q2_3

In [ ]:
import numpy as np
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import seaborn as sns


tukey = pairwise_tukeyhsd(
    endog=df_clean['price'],   # dependent variable: price
    groups=df_clean['name'],   # group variable: name
    alpha=0.05                 # significance level
)

print("\nTukey HSD results:")
print(tukey)                   # shows which pairs differ significantly

# If you want the Tukey table as a DataFrame:
tukey_df = pd.DataFrame(
    data=tukey._results_table.data[1:],   # skip header row
    columns=tukey._results_table.data[0]
)
print("\nTukey HSD as DataFrame:")
print(tukey_df)

# --------------------------------------------------
# 3-2) Visual check: compare mean price for each 'name'
#     using a bar plot + also a boxplot.
# --------------------------------------------------

# Bar plot of mean price by name (with confidence interval)
plt.figure(figsize=(12, 6))
sns.barplot(
    data=df_clean,
    x='name',
    y='price',
    estimator=np.mean,   # show mean price
    ci=95                # 95% confidence interval
)
plt.title("Mean price for each 'name'")
plt.ylabel("Mean price")
plt.xlabel("name")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Optional: Boxplot to see full distribution of prices by name
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_clean,
    x='name',
    y='price'
)
plt.title("Price distribution for each 'name'")
plt.ylabel("Price")
plt.xlabel("name")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Q2_4

In [ ]:
# ----------------------------------------------------
# 1) Remove rows with missing price
# ----------------------------------------------------
df_time = df_clean.copy()
df_time['year'] = 2018

df_time['datetime'] = pd.to_datetime(
    dict(
        year=df_time['year'],
        month=df_time['month'],
        day=df_time['day'],
        hour=df_time['hour']
    )
)

# ----------------------------------------------------
# 3) Sort by time
# ----------------------------------------------------
df_time = df_time.sort_values('datetime')

# ----------------------------------------------------
# 4) Aggregate price over time
#    Option A: average price per hour (fine-grained)
# ----------------------------------------------------
hourly_price = (
    df_time.set_index('datetime')
           .resample('h')['price']
           .mean()
)


# ----------------------------------------------------
# 5) Plot: price vs time
# ----------------------------------------------------
plt.figure(figsize=(14, 6))
plt.plot(hourly_price.index, hourly_price.values)
plt.xlabel('Time')
plt.ylabel('Average price')
plt.title('Change of ride price over time (hourly average)')
plt.tight_layout()
plt.show()


In [ ]:
daily_price = (
    df_time.set_index('datetime')
           .resample('D')['price']
           .mean()
)

plt.figure(figsize=(14,6))
plt.plot(daily_price.index, daily_price.values)
plt.title('Daily average ride price')
plt.xlabel('Date')
plt.ylabel('Average Price')
plt.show()

### Q2_5

In [ ]:
# ------------------------------------------------------
# 2) HISTOGRAM of price distribution
# ------------------------------------------------------
plt.figure(figsize=(10,5))
plt.hist(df_clean['price'], bins=40, edgecolor='black')
plt.title("Histogram of Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()

# ------------------------------------------------------
# 3) Detect outliers using IQR method
#    Outliers are values outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
# ------------------------------------------------------
Q1 = df_clean['price'].quantile(0.25)
Q3 = df_clean['price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter outliers
outliers = df_clean[(df_clean['price'] < lower_bound) | (df_clean['price'] > upper_bound)]

print("Number of outliers:", len(outliers))

# ------------------------------------------------------
# 4) HISTOGRAM of outliers only
# ------------------------------------------------------
plt.figure(figsize=(10,5))
plt.hist(outliers['price'], bins=30, edgecolor='black', color='red')
plt.title("Histogram of Price Outliers")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()

# ------------------------------------------------------
# 5) BOXPLOT to show distribution + outliers visually
# ------------------------------------------------------
plt.figure(figsize=(10,5))
sns.boxplot(x=df_clean['price'])
plt.title("Boxplot of Price (with outliers)")
plt.xlabel("Price")
plt.show()

### Q2_6

In [ ]:
import pandas as pd
from scipy.stats import shapiro, normaltest

# ----------------------------------------------
# 1) Clean dataset (remove missing price values)
# ----------------------------------------------
price_data = df_clean['price']

# ----------------------------------------------
# 2) Shapiro–Wilk Test
#    H0: Data is normally distributed
# ----------------------------------------------
shapiro_stat, shapiro_p = shapiro(price_data.sample(5000, random_state=0))
# Shapiro cannot handle very large samples, so we sample 5000 points
print("Shapiro–Wilk Test:")
print("Statistic:", shapiro_stat)
print("p-value:", shapiro_p)

# ----------------------------------------------
# 3) D’Agostino K² Test for Normality
#    H0: Data comes from a normal distribution
# ----------------------------------------------
dag_stat, dag_p = normaltest(price_data)
print("\nD’Agostino K² Test:")
print("Statistic:", dag_stat)
print("p-value:", dag_p)

# ----------------------------------------------
# 4) Interpretation based on p-values:
# ----------------------------------------------
def interpret(p):
    return "Reject normality (data is NOT normal)" if p < 0.05 else "Fail to reject normality (data may be normal)"

print("\nInterpretations:")
print("Shapiro–Wilk:", interpret(shapiro_p))
print("D’Agostino:", interpret(dag_p))


### Q2_7

In [ ]:
# Compute IQR
Q1 = df_clean['price'].quantile(0.25)
Q3 = df_clean['price'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Remove outliers
df_no_outliers_iqr = df_clean[(df_clean['price'] >= lower) & (df_clean['price'] <= upper)]

print("Original rows:", len(df_clean))
print("Rows after IQR outlier removal:", len(df_no_outliers_iqr))
print("Outliers removed:", len(df_clean) - len(df_no_outliers_iqr))


### Q2_8

In [ ]:
df_dt = df_clean.copy()   # or just df.copy() if you don't care about price here)
df_dt['year'] = 2018  # or the correct year for your data

df_dt['datetime'] = pd.to_datetime(
    dict(
        year=df_dt['year'],
        month=df_dt['month'],
        day=df_dt['day'],
        hour=df_dt['hour']
    )
)

df_dt['weekday'] = df_dt['datetime'].dt.dayofweek  # Monday=0, Sunday=6

# Optional: map weekday number to name for prettier plot
weekday_map = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
df_dt['weekday_name'] = df_dt['weekday'].map(weekday_map)

# -------------------------------------------------
# 2) Build a pivot table: number of trips by weekday & hour
# -------------------------------------------------
trip_counts = (
    df_dt
    .groupby(['weekday_name', 'hour'])
    .size()
    .reset_index(name='count')
)

# Pivot so rows = weekday, columns = hour, values = counts
heat_data = trip_counts.pivot(index='weekday_name', columns='hour', values='count')

# Reorder days for nicer display
heat_data = heat_data.reindex(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

# -------------------------------------------------
# 3) Plot heatmap
# -------------------------------------------------
plt.figure(figsize=(14, 6))
sns.heatmap(heat_data, annot=False)  # set annot=True if you want numbers in each cell
plt.title('Number of trips by weekday and hour')
plt.xlabel('Hour of day')
plt.ylabel('Day of week')
# plt.tight_layout()
plt.show()


# Q2_9

In [ ]:
plt.figure(figsize=(12, 6))
for weekday in ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']:
    subset = trip_counts[trip_counts['weekday_name'] == weekday]
    plt.plot(subset['hour'], subset['count'], label=weekday)

plt.title("Number of trips by weekday and hour")
plt.xlabel("Hour of day")
plt.ylabel("Number of trips")
plt.legend(title="Day of week", loc="upper right")
plt.xticks(range(24))
plt.tight_layout()
plt.show()

Cleaning Data for Q2_10

In [ ]:
#cab_type is categorical let's encode it to analyse its effect.
df_final = pd.get_dummies(df_final, columns=['cab_type'], prefix='cab', drop_first=True)
col = ['cab_Uber']
df_final[col] = df_final[col].astype(int)

#weekday_name (defined in Q2_8) is categorical let's encode it to analyse it's eefect
df_final['weekday_name'] = df_dt['weekday_name']
df_final = pd.get_dummies(df_final, columns=['weekday_name'], prefix='weekday', drop_first=True)
cols = [col for col in df_final.columns if col.startswith('weekday_')]
df_final[cols] = df_final[cols].astype(int)



In [ ]:
#checking skew for important variables
price_skew = df_final['price'].skew()
distance_skew = df_final['distance'].skew()

print(f"Skewness for Price: {price_skew:.4f}")
print(f"Skewness for Distance: {distance_skew:.4f}")

In [ ]:
#Transform accordingly
df_final['price'] = np.log(df_final['price'])
df_final['distance'] = np.sqrt(df_final['distance'])

In [ ]:
#Results of Transformartion
price_skew = df_final['price'].skew()
distance_skew = df_final['distance'].skew()

print(f"Skewness for Price: {price_skew:.4f}")
print(f"Skewness for Distance: {distance_skew:.4f}")

In [ ]:
categorical_cols = df_final.select_dtypes(exclude=[np.number]).columns
categorical_cols

# Q2_10

In [ ]:
import pandas as pd
import numpy as np

numerical_cols = df_final.select_dtypes(include=[np.number]).columns.tolist()

corr_matrix = df_final[numerical_cols].corr()

corr_matrix

In [ ]:
price_corr = corr_matrix.loc[: , 'price']
price_corr = price_corr.drop('price')
sorted_corr = price_corr.sort_values(ascending=False)
sorted_corr

Drop unnecessary columns(explained in document)

In [ ]:
columns_to_drop = ['source', 'destination', 'latitude', 'longitude', 'weekday_Mon', 'weekday_Sat', 'weekday_Sun', 'weekday_Thu', 'weekday_Tue', 'weekday_Wed', 'hour', 'day', 'month']
df_final = df_final.drop(columns=columns_to_drop, errors='ignore')

# Q3_1 PCA

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. جداسازی ستون‌های آب و هوا
weather_cols = [
    'temperature', 'apparentTemperature', 'precipIntensity', 'precipProbability',
    'humidity', 'windSpeed', 'windGust', 'visibility', 'temperatureHigh',
    'temperatureLow', 'apparentTemperatureHigh', 'apparentTemperatureLow',
    'dewPoint', 'pressure', 'windBearing', 'cloudCover', 'uvIndex', 'ozone',
    'moonPhase', 'temperatureMin', 'temperatureMax', 'apparentTemperatureMin', 'apparentTemperatureMax'
] # لیست کامل ستون‌های آب و هوا
weather_data = df_final[weather_cols].copy()

# 2. استانداردسازی داده‌ها
scaler = StandardScaler()
weather_scaled = scaler.fit_transform(weather_data)

# اجرای PCA با تعداد مولفه‌های برابر تعداد ویژگی‌ها
pca = PCA()
pca.fit(weather_scaled)

# محاسبه واریانس تجمعی
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

# یافتن تعداد مولفه‌هایی که 90% واریانس را پوشش می‌دهند
n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1
print(f"components needed to cover 90% of the variance: {n_components_90}")

# اجرای نهایی PCA
pca_final = PCA(n_components=n_components_90)
weather_pca = pca_final.fit_transform(weather_scaled)
weather_pca_df = pd.DataFrame(weather_pca,
                              columns=[f'Weather_PC_{i+1}' for i in range(n_components_90)],
                              index=df_final.index)

# ادغام مؤلفه‌های PCA با دیتافریم نهایی
df_final1 = pd.concat([df_final.drop(columns=weather_cols, errors='ignore'), weather_pca_df], axis=1)

In [ ]:
# 1. Extract the Eigenvectors (Loadings)
# The components_ attribute holds the matrix of eigenvectors.
loadings = pca_final.components_

# 2. Convert to DataFrame for better readability
# .T is used to transpose the matrix so that features are rows and PCs are columns.
loading_df = pd.DataFrame(
    loadings.T,
    columns=[f'Weather_PC_{i+1}' for i in range(n_components_90)],
    index=weather_cols
)

# 3. Display Top 5 features contributing to each PC
print("Feature Loadings (Contributions) to Principal Components:")

for pc in loading_df.columns:
    # Sort by the absolute value to find the most influential features
    top_features = loading_df[pc].abs().sort_values(ascending=False).head(5)

    print(f"\n--- {pc} (Top 5 Influencers) ---")

    # Print the actual loading values for the top features
    print(loading_df.loc[top_features.index, [pc]])

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np 

weather_cols = [ # Full list of weather columns
    'temperature', 'apparentTemperature', 'precipIntensity', 'precipProbability',
    'humidity', 'windSpeed', 'windGust', 'visibility', 'temperatureHigh',
    'temperatureLow', 'apparentTemperatureHigh', 'apparentTemperatureLow',
    'dewPoint', 'pressure', 'windBearing', 'cloudCover', 'uvIndex', 'ozone',
    'moonPhase', 'temperatureMin', 'temperatureMax', 'apparentTemperatureMin', 'apparentTemperatureMax'
]
weather_data = df_final[weather_cols].copy()
scaler = StandardScaler()
weather_scaled = scaler.fit_transform(weather_data)

# 2. Determine Optimal Number of Clusters (k) using the Elbow Method
print("Running K-Means and determining optimal k (Elbow Plot):")
model = KMeans(random_state=42, n_init='auto')
# Test k values between 2 and 10
visualizer = KElbowVisualizer(model, k=(2,10), timings=False)
visualizer.fit(weather_scaled)

optimal_k = visualizer.elbow_value_ if visualizer.elbow_value_ is not None else 4

# 3. Final K-Means Execution
print(f"Optimal number of clusters (k): {optimal_k}")
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
# Create the new Cluster Label column in the original dataframe
df_final['Weather_KMeans_Cluster'] = kmeans.fit_predict(weather_scaled)

# 4. Preparation for Modeling: One-Hot Encoding
# Since the cluster labels are categorical variables, they must be One-Hot Encoded
df_final_kmeans = pd.get_dummies(df_final, columns=['Weather_KMeans_Cluster'], prefix='KMC', drop_first=True)

# df_final_kmeans now contains the new weather cluster variables
print("New K-Means clustering columns have been added to the dataframe.")

# Q3_2 & Q3_3

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

df_final['is_raining'] = (df_final['precipIntensity'] > 0).astype(int)
weather_features = [
    'temperature', 'apparentTemperature', 'precipIntensity', 'precipProbability',
    'humidity', 'windSpeed', 'windGust', 'visibility', 'temperatureHigh',
    'temperatureLow', 'apparentTemperatureHigh', 'apparentTemperatureLow',
    'dewPoint', 'pressure', 'windBearing', 'cloudCover', 'uvIndex', 'ozone',
    'moonPhase', 'temperatureMin', 'temperatureMax', 'apparentTemperatureMin',
    'apparentTemperatureMax'
]
X_cls = df_final[weather_features].copy()
Y_cls = df_final['is_raining']

# Dropping the target-defining column itself from X to avoid data leakage
X_cls = X_cls.drop(columns=['precipIntensity', 'precipProbability'])

# Split data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(
    X_cls, Y_cls, test_size=0.3, random_state=42, stratify=Y_cls
)
numerical_features = X_cls.select_dtypes(include=np.number).columns.tolist()
scaler = StandardScaler()
X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test[numerical_features] = scaler.transform(X_test[numerical_features])


Model 1: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Train the model
log_reg = LogisticRegression(solver='liblinear', random_state=42)
log_reg.fit(X_train, Y_train)

# Predict on the test set
Y_pred_log = log_reg.predict(X_test)

print("--- Logistic Regression Results ---")
print(f"Accuracy: {accuracy_score(Y_test, Y_pred_log):.4f}")
print("Classification Report:")
print(classification_report(Y_test, Y_pred_log))

Logistic Regression Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate the confusion matrix
cm_log = confusion_matrix(Y_test, Y_pred_log)

# Plotting the Confusion Matrix


plt.figure(figsize=(6, 4))
sns.heatmap(cm_log, annot=True, fmt='d', cmap='Greys',
            xticklabels=['No Rain (0)', 'Rain (1)'],
            yticklabels=['No Rain (0)', 'Rain (1)'])
plt.title('Logistic Regression Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

print("\nLogistic Regression Confusion Matrix:\n", cm_log)

Model 2: Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Train the model
dt_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_clf.fit(X_train, Y_train)

# Predict on the test set
Y_pred_dt = dt_clf.predict(X_test)

print("\n--- Decision Tree Results ---")
print(f"Accuracy: {accuracy_score(Y_test, Y_pred_dt):.4f}")
print("Classification Report:")
print(classification_report(Y_test, Y_pred_dt))

Decision Tree Confusion Matrix

In [ ]:
# Generate the confusion matrix
cm_dt = confusion_matrix(Y_test, Y_pred_dt)

# Plotting the Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greys',
            xticklabels=['No Rain (0)', 'Rain (1)'],
            yticklabels=['No Rain (0)', 'Rain (1)'])
plt.title('Decision Tree Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

print("\nDecision Tree Confusion Matrix:\n", cm_dt)

In [ ]:
numerical_cols = df_final1.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df_final1[numerical_cols].corr()

# Q4_1

In [ ]:
# 2. Define X and Y
Y = df_final1['price'] # Already log-transformed
X = df_final1[numerical_cols].copy()
X = X.drop(columns=['price'])

# 3. Handle categorical features (One-Hot Encoded features are already in X)
# Standardize numerical columns in X before Wrapper and Embedded methods
from sklearn.preprocessing import StandardScaler
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()

scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# Convert X back to a DataFrame after scaling to keep feature names
X_scaled = pd.DataFrame(X, columns=X.columns, index=X.index)

Method 1: Correlation Filter

In [ ]:
# Create a DataFrame containing all features + the target
data_corr = X.copy()
data_corr['price'] = Y

# Calculate correlation matrix with the target
corr_with_price = data_corr.corr()['price'].sort_values(ascending=False)

# Select features based on a threshold (e.g., absolute correlation > 0.05)
# Note: For Log-Price, even small correlations like 0.05 can be significant.
correlation_threshold = 0.05
selected_corr_features = corr_with_price[abs(corr_with_price) >= correlation_threshold].index.tolist()

# Remove the target variable itself from the list
selected_corr_features.remove('price')

print(f"\nMethod 1 (Correlation) Selected Features ({len(selected_corr_features)}):")
print(selected_corr_features)

Method 2: Backward Elimination (using feature importance)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

# Initialize the model (Wrapper)
estimator = LinearRegression()

# Initialize RFE, aiming to select a specific number of features (e.g., 10 features)
# The ideal number should be chosen iteratively, but for demonstration, we set k=10
k_features = 10
rfe = RFE(estimator, n_features_to_select=k_features, step=1)
rfe.fit(X_scaled, Y)

# Get the selected features
selected_rfe_features = X_scaled.columns[rfe.support_].tolist()

print(f"\nMethod 2 (RFE/Wrapper) Selected Features ({k_features}):")
print(selected_rfe_features)

Method 3: Lasso Embedded Method

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold

# Use Cross-Validation to find the optimal alpha (regularization strength)
lasso = LassoCV(
    cv=KFold(5),
    random_state=42,
    n_jobs=-1,
    max_iter=5000 # Increase max_iter for convergence
).fit(X_scaled, Y)

# Identify features with non-zero coefficients
lasso_coefficients = pd.Series(lasso.coef_, index=X_scaled.columns)
selected_lasso_features = lasso_coefficients[lasso_coefficients != 0].index.tolist()

print(f"\nMethod 3 (Lasso) Selected Features ({len(selected_lasso_features)}):")
print(selected_lasso_features)

# Q4_2


In [ ]:
#Fitting a regression model on variables from method 1(correlation)
Target = 'price'
Predictors_1 = ['name_Lux Black XL', 'name_Black SUV', 'distance', 'name_Lux Black',
              'surge_multiplier', 'name_Lux', 'name_UberX', 'name_WAV', 'name_Lyft', 'name_UberPool', 'name_Shared']
y = df_final1[Target]
x_1 = df_final1[Predictors_1]


In [ ]:
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

x_train, x_test, y_train, y_test = train_test_split(
    x_1, y, test_size=0.2)
x_train_const = sm.add_constant(x_train)
x_test_const  = sm.add_constant(x_test)

model = sm.OLS(y_train, x_train_const).fit()
print(model.summary())

In [ ]:
#معیار دوم مقایسه برای متد اول
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
y_pred = model.predict(x_test_const)
rmse_1 = np.sqrt(mean_squared_error(y_test, y_pred))
print(rmse_1)

In [ ]:
#معیار سوم مقایسه برای متد اول
from sklearn.metrics import median_absolute_error
med_ae_1 = median_absolute_error(y_test, y_pred)
print(med_ae_1)

In [ ]:
#Fitting a regression model on variables from method 3 (Lasso)
Predictors_3 = ['distance', 'surge_multiplier', 'name_Black SUV', 'name_Lux Black', 'name_Lux Black XL', 'name_Lyft', 'name_Lyft XL',
              'name_Shared', 'name_UberPool', 'name_UberX', 'name_UberXL', 'name_WAV', 'cab_Uber']
x_3 = df_final1[Predictors_3]
x_train, x_test, y_train, y_test = train_test_split(
    x_3, y, test_size=0.2)
x_train_const = sm.add_constant(x_train)
x_test_const  = sm.add_constant(x_test)
model = sm.OLS(y_train, x_train_const).fit()
print(model.summary())


In [ ]:
#معیار دوم مقایسه برای متد سوم
y_pred = model.predict(x_test_const)
rmse_3 = np.sqrt(mean_squared_error(y_test, y_pred))
print(rmse_3)

In [ ]:
#معیار سوم مقایسه برای متد سوم
med_ae_3= median_absolute_error(y_test, y_pred)
print(med_ae_3)

# Q4_3 AdaBoost

In [ ]:
#AdaBoost
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
stump = DecisionTreeRegressor(max_depth = 2, random_state = 42)
adaboost_model = AdaBoostRegressor(
    estimator = stump,
    n_estimators = 200,
    learning_rate = 0.5,
    random_state = 42
)
adaboost_model.fit(x_3, y)
R2 = adaboost_model.score(x_3, y)
print (R2)

# Q4_5 Optimizing AdaBoost's Hyperparameters using Grid search

In [ ]:
#Q4_5
#Optimizing AdaBoost's Hyperparameters using Grid search
from sklearn.model_selection import GridSearchCV
ada = AdaBoostRegressor(estimator=stump, random_state=42)

rows = np.random.choice(len(x_3), size=3000, replace=False)
X_small = x_3.iloc[rows]
y_small = y.iloc[rows]

param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.1, 0.5, 1.0],
    'estimator__max_depth': [2, 4, 6]
}
grid = GridSearchCV(
    estimator=ada,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_small, y_small)

print("Best params (Grid Search):", grid.best_params_)
print("Best R2:", grid.best_score_)



### Q4_5 Taguchi trial using XGBoost

The submitted heading named AdaBoost. The implementation in this cell fits XGBoost.

In [ ]:
#Q4_5
# Taguchi trial using XGBoost

import xgboost as xgb
sample_idx = np.random.choice(len(x_3), 3000, replace=False)
X = x_3.iloc[sample_idx]
Y = y.iloc[sample_idx]

x_train, x_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

max_depth_levels = [2,4 ,6 ]
learning_rate_levels = [0.1, 0.5, 1]
n_estimators_levels = [100, 200, 50]


L9 = np.array([
    [1,1,1,1],
    [1,2,2,2],
    [1,3,3,3],
    [2,1,2,3],
    [2,2,3,1],
    [2,3,1,2],
    [3,1,3,2],
    [3,2,1,3],
    [3,3,2,1],
])

results = []

for row in L9:
    params = {
        "max_depth": max_depth_levels[row[0]-1],
        "learning_rate": learning_rate_levels[row[1]-1],
        "n_estimators": n_estimators_levels[row[2]-1],
        "objective": "reg:squarederror",
        "random_state": 42
    }

    model = xgb.XGBRegressor(**params)
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    r2 = r2_score(y_test, pred)

    results.append((params, r2))



results = sorted(results, key=lambda x: x[1], reverse=True)

print("\nBest Taguchi configuration:")
print(results[0])


# Q4_3 GXBoost

In [ ]:
#GXBoost
import xgboost as xgb
from sklearn.metrics import r2_score
x_train, x_test, y_train, y_test = train_test_split(x_3, y, test_size=0.2, random_state=42)
xgb_parameters = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.1,
    'max_depth': 3,
    'n_estimators': 300,
    'random_state': 42
}
xgb_model = xgb.XGBRegressor(**xgb_parameters)
xgb_model.fit(x_train, y_train)
y_pred = xgb_model.predict(x_test)
r2 = r2_score(y_test, y_pred)
print (r2)

# Q4_5 Optimizing XGBoost Hyperparameters using Grid search

In [ ]:
#Q4_5
#Optimizing XGBoost Hyperparameters using Grid search
rows = np.random.choice(len(x_train), size=3000, replace=False)
X_small = x_train.iloc[rows]
y_small = y_train.iloc[rows]

param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0]
}

grid = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_small, y_small)

print("Best params (Grid Search):", grid.best_params_)
print("Best R2 (Grid Search):", grid.best_score_)


# Q4_5 Optimizing XGBoost HyperParameters using Taguchi

In [ ]:
#Q4_5
#Optimizing XGBoost HyperParameters using Taguchi


x_small = x_3.iloc[:3000].copy()
y_small = y.iloc[:3000].copy()

x_train, x_test, y_train, y_test = train_test_split(
    x_small, y_small, test_size=0.2, random_state=42
)


taguchi_L9 = [
    [0.05, 3, 100],
    [0.05, 4, 200],
    [0.05, 6, 400],
    [0.10, 3, 200],
    [0.10, 4, 400],
    [0.10, 6, 100],
    [0.20, 3, 400],
    [0.20, 4, 100],
    [0.20, 6, 200]
]

results = []

for i, (lr, depth, nest) in enumerate(taguchi_L9, start=1):
    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        learning_rate=lr,
        max_depth=depth,
        n_estimators=nest,
        random_state=42
    )

    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    r2 = r2_score(y_test, y_pred)

    results.append([i, lr, depth, nest, r2])
    print(f"Experiment {i} → R2 = {r2:.4f}")

df_results = pd.DataFrame(
    results,
    columns=["Exp", "learning_rate", "max_depth", "n_estimators", "R2"]
)

print("\nTaguchi Results:")
print(df_results)
print("\nBest combination:")
print(df_results.loc[df_results["R2"].idxmax()])


# Q4_6


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import StackingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

Predictors_3 = [
    'distance', 'surge_multiplier', 'name_Black SUV', 'name_Lux Black', 'name_Lux Black XL',
    'name_Lyft', 'name_Lyft XL', 'name_Shared', 'name_UberPool', 'name_UberX',
    'name_UberXL', 'name_WAV', 'cab_Uber'
]

X_stk = df_final1[Predictors_3]
Y_stk = df_final1['price']

X_train_stk, X_test_stk, Y_train_stk, Y_test_stk = train_test_split(
    X_stk, Y_stk, test_size=0.2, random_state=42
)

estimators = [
    ('OLS', LinearRegression()), # مدل خطی OLS
    ('XGB', XGBRegressor(
        objective='reg:squarederror', learning_rate=0.1, max_depth=3,
        n_estimators=300, random_state=42, n_jobs=-1
    )),
    ('Ada', AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=6), # تنظیمات بهینه AdaBoost از GridSearch شما
        n_estimators=200, learning_rate=0.1, random_state=42
    ))
]

# تعریف مدل ترکیبی (Stacking Regressor)
# Meta-Model: Ridge Regression (جهت حفظ سادگی و اجتناب از بیش‌برازش)
stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5, # استفاده از 5-fold Cross-Validation
    n_jobs=-1
)


print("Training Stacking Ensemble Model...")
stacking_model.fit(X_train_stk, Y_train_stk)

Y_pred_stk = stacking_model.predict(X_test_stk)
r2_stk = r2_score(Y_test_stk, Y_pred_stk)
rmse_stk = np.sqrt(mean_squared_error(Y_test_stk, Y_pred_stk))
mae_stk = mean_absolute_error(Y_test_stk, Y_pred_stk)

print("\n--- Stacking Ensemble Model Results (Final Model) ---")
print(f"R-squared (R2): {r2_stk:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_stk:.4f}")
print(f"Mean Absolute Error (MAE): {mae_stk:.4f}")